# Load a model and chat

This standalone tutorial loads one Hugging Face model, sends messages, and releases it. No earlier tutorial needs to be run. RAG is not used yet.

On a new computer, first run the setup command in the README. Select QAChat's `.venv` kernel, then run one cell at a time with Shift+Enter.

In [ ]:
from pathlib import Path
import platform
import sys

root = Path.cwd()
if not (root / "qachat/chat.py").is_file():
    root = root.parent
assert (root / "qachat/chat.py").is_file()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
assert Path(sys.prefix).resolve() == (root / ".venv").resolve()

from qachat.chat import ChatSession
from qachat.huggingface import DEFAULT_MODELS
from qachat.huggingface import available_models

print("Python:", sys.executable)
print("Computer:", platform.system(), platform.machine())

## 1. See models already cached

This reads Hugging Face's shared cache. A model downloaded by another Hugging Face application can appear here.

Each printed name is a **model ID**. Copy the model ID you want into `MODEL_ID` in the next section. For example: `MODEL_ID = "lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly"`.

This is a list of cached **model candidates**, not a list of guaranteed chat models. The cache may also contain embedding, image, or incompatible models. Choose a text-generation/chat model whose weights match MLX or PyTorch.

In [ ]:
models = available_models()
if models:
    print("Model IDs found in the Hugging Face cache:")
    for number, model in enumerate(models, start=1):
        size_gb = model["size_bytes"] / 1_000_000_000
        print(f"{number}. {model['model_id']} ({size_gb:.2f} GB)")
else:
    print("No models are cached yet.")

## 2. Choose the model and engine

The simplest choice is:

```python
BACKEND = "auto"
MODEL_ID = None
```

`None` means **use QAChat's default model**. On Apple Silicon, `auto` selects MLX and defaults to `mlx-community/Qwen3-0.6B-4bit`. On other supported computers, `auto` selects PyTorch and defaults to `Qwen/Qwen3-0.6B`.

To use another cached MLX chat model, copy its exact ID from the list above:

```python
BACKEND = "mlx"
MODEL_ID = "mlx-community/Qwen3-14B-4bit"
```

For a Transformers/PyTorch model, use `BACKEND = "torch"` and its exact model ID. `auto` chooses an engine from the computer; it does not inspect or convert custom model weights. Do not select the cached embedding model for chat.

To change QAChat's project-wide defaults, edit the two values in `DEFAULT_MODELS` in [qachat/huggingface.py](../qachat/huggingface.py). Normally, setting `MODEL_ID` here is safer because it changes only this session.

`THINKING` is passed to compatible chat templates. Temperature `0` gives repeatable greedy output. `MAX_NEW_TOKENS = None` adds no QAChat length cap; generation uses the model's remaining context. Set a positive number for a shorter experiment.

If `ALLOW_DOWNLOAD` is true and files are missing, Hugging Face downloads them to its normal cache during the loading cell.

In [ ]:
BACKEND = "auto"
MODEL_ID = None

is_apple_silicon = (
    platform.system() == "Darwin"
    and platform.machine().lower() == "arm64"
)
selected_backend = (
    "mlx" if BACKEND == "auto" and is_apple_silicon else BACKEND
)
if selected_backend == "auto":
    selected_backend = "torch"
default_model = DEFAULT_MODELS[selected_backend]
chosen_model = MODEL_ID or default_model
print("Default model:", default_model)
print("Model selected for this chat:", chosen_model)

THINKING = False
TEMPERATURE = 0.0
MAX_NEW_TOKENS = None
ALLOW_DOWNLOAD = True

## 3. Load the selected model

This is the only cell that loads weights. Loading can use several GB of memory. It closes an earlier session before creating a new one.

In [ ]:
if "session" in globals():
    session.close()

session = ChatSession(
    MODEL_ID,
    backend=BACKEND,
    thinking=THINKING,
    temperature=TEMPERATURE,
    max_new_tokens=MAX_NEW_TOKENS,
    system_prompt="Answer clearly and briefly.",
    allow_download=ALLOW_DOWNLOAD,
)
print("Loaded model:", session.model_id)
print("Engine and device:", session.backend, session.device)

## 4. Chat

The second message receives the earlier turn as conversation history. Change the text and rerun either cell to continue.

In [ ]:
answer = session.reply(
    "Hello, I am Iok-ui! Greet me in one short sentence."
)
assert answer.strip()
print(answer)

In [ ]:
answer = session.reply(
    "Copy the name from my first message. Answer with the name only."
)
assert "Iok-ui" in answer
print(answer)

## 5. Release the model

Closing releases this session's model memory and clears its messages. Cached model files remain available for later use.

In [ ]:
session.close()
print("Chat closed.")